# FBNetGen v5 Full Training — All 25 Folds

Trains FBNetGen with best Optuna hyperparameters on all 25 folds (5x5 nested CV).

Best config from search: `avg_val_r = 0.4799` (trial 46)

### Before running
1. **Runtime → Change runtime type → A100/H100 GPU**
2. Upload `folds_data/` (or `MRI_data/`) to Google Drive
3. Upload `GNN-mri/` project folder to Drive
4. Edit the **CONFIG cell** below
5. Run all cells

In [ ]:
# ================================================================
# CONFIG
# ================================================================
DRIVE_FOLD_DIR   = '/content/drive/MyDrive/MRI_data'

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/GNN-mri'

# Output — results saved to Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/results/fbnetgen_v5_full'

# Best hyperparameters from Optuna search (trial 46)
CONFIG = {
    'lr': 0.000479000672011449,
    'weight_decay': 1.6308226683675615e-05,
    'batch_size': 64,
    'hidden_dim': 128,
    'dropout': 0.25248380464423814,
    'n_layers': 3,
    'n_heads': 2,
    'refine_graph': False,
    'epochs': 100,
    'patience': 20,
    'seed': 42,
    'normalize_method': 'standard',
    'scheduler': 'cosine',
    'n_rois': 268,
}

## Step 1 — Mount Drive + Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
drive_out = Path(DRIVE_OUTPUT_DIR)
drive_out.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {drive_out}')

import subprocess, torch

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])

torch_ver = torch.__version__.split('+')[0]
cuda_ver  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_ver}  |  CUDA build: {cuda_ver}')

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse -f {pyg_url}')
run('pip install -q scikit-learn scipy dill')
print('Dependencies ready.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/results/fbnetgen_v5_full
PyTorch 2.10.0  |  CUDA build: cu128
Dependencies ready.


## Step 2 — Setup project + copy data

In [ ]:
import os, shutil, sys

PROJECT_ROOT = Path('/content/GNN-mri')
src = Path(DRIVE_PROJECT_DIR)
assert src.exists(), f'Project not found at {src}'

if not PROJECT_ROOT.exists():
    print('Copying project from Drive...')
    shutil.copytree(str(src), str(PROJECT_ROOT))
else:
    print('Project already at /content/GNN-mri')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Copy fold data to local disk (faster I/O)
LOCAL_FOLD_DIR = Path('/content/folds_data')
DRIVE_FOLD_PATH = Path(DRIVE_FOLD_DIR)
assert DRIVE_FOLD_PATH.exists(), f'Fold data not found at {DRIVE_FOLD_PATH}'

if not LOCAL_FOLD_DIR.exists():
    print('Copying fold data to local disk...')
    shutil.copytree(str(DRIVE_FOLD_PATH), str(LOCAL_FOLD_DIR))
else:
    print('Fold data already on local disk.')

fold_files = sorted(LOCAL_FOLD_DIR.glob('graphs_outer*.pkl'))
print(f'Found {len(fold_files)} fold files.')

Project already at /content/GNN-mri
Copying fold data to local disk...
Found 5 fold files.


## Step 3 — GPU setup + imports

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, json, random, time
from datetime import datetime
from torch.optim.lr_scheduler import CosineAnnealingLR

from models.fbnetgen import FBNetGenFromGraph
from utils.data_utils import (
    load_graphs_with_normalization,
    create_dataloaders,
    compute_metrics,
    aggregate_window_predictions,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark = True
    print('TF32 + high matmul precision + cuDNN benchmark enabled')
else:
    print('WARNING: No GPU')

AMP_DTYPE = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16
use_bf16 = (AMP_DTYPE == torch.bfloat16)
print(f'Device: {device}, AMP: {AMP_DTYPE}')

GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
TF32 + high matmul precision + cuDNN benchmark enabled
Device: cuda, AMP: torch.bfloat16


## Step 4 — Training functions

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    n = 0
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=True, dtype=AMP_DTYPE):
            preds = model(batch)
            loss = criterion(preds, batch.y.float())
        if use_bf16:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return total_loss / max(1, n)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_targets, all_sids = [], [], []
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=True, dtype=AMP_DTYPE):
            preds = model(batch)
        all_preds.append(preds.float().cpu())
        all_targets.append(batch.y.cpu())
        if hasattr(batch, 'subject_id'):
            all_sids.append(batch.subject_id.cpu())

    preds_np = torch.cat(all_preds).numpy()
    targets_np = torch.cat(all_targets).numpy()
    metrics = compute_metrics(preds_np, targets_np)

    subj_r = None
    if all_sids:
        sids = torch.cat(all_sids).numpy().flatten()
        sp, _ = aggregate_window_predictions(preds_np, sids)
        st, _ = aggregate_window_predictions(targets_np, sids)
        subj_metrics = compute_metrics(sp, st)
        metrics['subj_r'] = subj_metrics.get('r', 0)
        metrics['subj_mse'] = subj_metrics.get('mse', 0)
        metrics['subj_mae'] = subj_metrics.get('mae', 0)

    return metrics, preds_np, targets_np

print('Training functions defined.')

Training functions defined.


## Step 5 — Train all 25 folds

In [ ]:
all_results = []
total_start = time.time()

for fold_idx, fold_path in enumerate(fold_files):
    fold_name = fold_path.stem
    print(f'\n{"="*60}')
    print(f'Fold {fold_idx+1}/{len(fold_files)}: {fold_name}')
    print(f'{"="*60}')

    set_seed(CONFIG['seed'])
    fold_start = time.time()

    # Load data
    train_graphs, val_graphs, test_graphs, info = load_graphs_with_normalization(
        str(fold_path), normalize_method=CONFIG['normalize_method']
    )
    train_loader, val_loader, test_loader = create_dataloaders(
        train_graphs, val_graphs, test_graphs,
        batch_size=CONFIG['batch_size'], num_workers=0, pin_memory=True,
    )

    # Build model
    in_dim = train_graphs[0].x.size(-1)
    model = FBNetGenFromGraph(
        in_dim=in_dim,
        hidden_dim=CONFIG['hidden_dim'],
        n_layers=CONFIG['n_layers'],
        n_heads=CONFIG['n_heads'],
        dropout=CONFIG['dropout'],
        refine_graph=CONFIG['refine_graph'],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')

    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    criterion = nn.MSELoss()
    scaler = None if use_bf16 else torch.amp.GradScaler('cuda')

    # LR schedule: warmup + cosine
    n_warmup = max(1, CONFIG['epochs'] // 5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=n_warmup),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CONFIG['epochs'] - n_warmup), eta_min=CONFIG['lr'] * 0.01),
        ],
        milestones=[n_warmup],
    )

    best_val_r = -float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(CONFIG['epochs']):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        scheduler.step()

        val_metrics, _, _ = evaluate(model, val_loader)
        val_r = val_metrics.get('subj_r', val_metrics.get('r', 0))

        if val_r > best_val_r + 1e-5:
            best_val_r = val_r
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CONFIG['patience']:
            print(f'  Early stop at epoch {epoch+1}')
            break

    # Load best and evaluate on test
    model.load_state_dict(best_state)
    test_metrics, test_preds, test_targets = evaluate(model, test_loader)
    fold_elapsed = time.time() - fold_start

    test_r = test_metrics.get('subj_r', test_metrics.get('r', 0))
    test_mse = test_metrics.get('subj_mse', test_metrics.get('mse', 0))
    print(f'  val_r={best_val_r:.4f}  test_r={test_r:.4f}  test_mse={test_mse:.4f}  [{fold_elapsed:.0f}s]')

    # Save checkpoint to Drive
    fold_out = drive_out / fold_name
    fold_out.mkdir(parents=True, exist_ok=True)
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': best_state,
        'config': CONFIG,
        'val_metrics': {'subj_r': best_val_r},
        'test_metrics': test_metrics,
    }, fold_out / 'fbnetgen_best.pt')

    # Save predictions
    with open(fold_out / 'fbnetgen_predictions.json', 'w') as f:
        json.dump({
            'test_predictions': test_preds.tolist(),
            'test_targets': test_targets.tolist(),
            'test_metrics': {k: float(v) for k, v in test_metrics.items()},
        }, f, indent=2)

    all_results.append({
        'fold': fold_name,
        'val_r': float(best_val_r),
        'test_r': float(test_r),
        'test_mse': float(test_mse),
        'stopped_epoch': epoch + 1,
        'time_seconds': fold_elapsed,
    })

    # Free memory
    del model, optimizer, scheduler, train_loader, val_loader, test_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_elapsed = time.time() - total_start
total_mins = int(total_elapsed // 60)
print(f'\nAll {len(fold_files)} folds done in {total_mins}m {int(total_elapsed % 60)}s')


Fold 1/5: graphs_outer1_inner1
Loading fold from /content/folds_data/graphs_outer1_inner1.pkl
Target normalization (standard):
  Original range: [66.49, 132.49]
  Normalized range: [-2.73, 2.35]
  Mean: 0.0000, Std: 1.0000
#train graphs (windows): 10440
#val   graphs (windows): 2700
#test  graphs (windows): 3330
  Parameters: 210,050
  Early stop at epoch 28
  val_r=0.5068  test_r=0.4858  test_mse=0.9661  [2483s]

Fold 2/5: graphs_outer1_inner2
Loading fold from /content/folds_data/graphs_outer1_inner2.pkl
Target normalization (standard):
  Original range: [66.49, 132.49]
  Normalized range: [-2.55, 2.23]
  Mean: -0.0000, Std: 1.0000
#train graphs (windows): 10530
#val   graphs (windows): 2610
#test  graphs (windows): 3330
  Parameters: 210,050
  Early stop at epoch 27
  val_r=0.4637  test_r=0.4900  test_mse=0.9414  [2381s]

Fold 3/5: graphs_outer1_inner3
Loading fold from /content/folds_data/graphs_outer1_inner3.pkl
Target normalization (standard):
  Original range: [66.49, 131.78]
 

## Step 6 — Aggregate results

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

print('='*60)
print('AGGREGATE RESULTS: FBNetGen v5 Full (all 25 folds)')
print('='*60)
print(f'\nTest subject-level Pearson r:')
print(f'  Mean:   {df["test_r"].mean():.4f} +/- {df["test_r"].std():.4f}')
print(f'  Median: {df["test_r"].median():.4f}')
print(f'  Min:    {df["test_r"].min():.4f}')
print(f'  Max:    {df["test_r"].max():.4f}')
print(f'\nTest MSE:')
print(f'  Mean:   {df["test_mse"].mean():.4f} +/- {df["test_mse"].std():.4f}')
print(f'\nVal r:')
print(f'  Mean:   {df["val_r"].mean():.4f} +/- {df["val_r"].std():.4f}')

# Per outer fold summary (5 outer folds x 5 inner folds)
print(f'\n--- Per outer fold ---')
for outer in range(1, 6):
    outer_df = df[df['fold'].str.contains(f'outer{outer}_')]
    if not outer_df.empty:
        print(f'  Outer {outer}: test_r = {outer_df["test_r"].mean():.4f} +/- {outer_df["test_r"].std():.4f}')

print(f'\n--- Full table ---')
print(df[['fold', 'val_r', 'test_r', 'test_mse', 'stopped_epoch']].to_string(index=False))

# Save aggregate summary
summary = {
    'model': 'fbnetgen',
    'config': CONFIG,
    'n_folds': len(all_results),
    'test_r_mean': float(df['test_r'].mean()),
    'test_r_std': float(df['test_r'].std()),
    'test_mse_mean': float(df['test_mse'].mean()),
    'val_r_mean': float(df['val_r'].mean()),
    'per_fold': all_results,
    'timestamp': datetime.now().isoformat(),
}
with open(drive_out / 'fbnetgen_aggregate_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

df.to_csv(drive_out / 'fbnetgen_all_folds.csv', index=False)
print(f'\nResults saved to {drive_out}/')

AGGREGATE RESULTS: FBNetGen v5 Full (all 25 folds)

Test subject-level Pearson r:
  Mean:   0.4755 +/- 0.0371
  Median: 0.4858
  Min:    0.4107
  Max:    0.5052

Test MSE:
  Mean:   0.9581 +/- 0.0368

Val r:
  Mean:   0.4703 +/- 0.0424

--- Per outer fold ---
  Outer 1: test_r = 0.4755 +/- 0.0371

--- Full table ---
                fold    val_r   test_r  test_mse  stopped_epoch
graphs_outer1_inner1 0.506793 0.485779  0.966061             28
graphs_outer1_inner2 0.463739 0.490032  0.941393             27
graphs_outer1_inner3 0.503225 0.485747  0.903279             25
graphs_outer1_inner4 0.475813 0.505163  0.995368             28
graphs_outer1_inner5 0.401761 0.410719  0.984316             40

Results saved to /content/drive/MyDrive/results/fbnetgen_v5_full/
